In [1]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torchvision.io import decode_image
from torchvision.models import get_model, get_model_weights
from torchcam.methods import LayerCAM
from torchcam.utils import overlay_mask
from torchvision.transforms.functional import to_pil_image

weights = get_model_weights("vgg16").DEFAULT
model = get_model("vgg16", weights=weights).eval()
preprocess = weights.transforms()
categories = weights.meta["categories"]

layers = ["features.2", "features.14", "features.28"]
layer_names = [
    "Early (features.2)",
    "Middle (features.14)",
    "Late (features.28)"
]

def load_image(image_path):
    img = decode_image(image_path)
    input_tensor = preprocess(img).unsqueeze(0).requires_grad_(True)
    return img, input_tensor

def predict_top5(input_tensor):
    with torch.enable_grad():
        out = model(input_tensor)
        probs = F.softmax(out.squeeze(0), dim=0)
    class_idx = probs.argmax().item()
    confidence = probs[class_idx].item() * 100
    top5 = torch.topk(probs, 5)
    return class_idx, confidence, top5

def make_cam_overlay(img, image_path, class_idx, target_layer, alpha=0.3):
    _, fresh_tensor = load_image(image_path)  # färsk tensor varje gång

    cam_extractor = LayerCAM(model, target_layer=target_layer)

    with torch.enable_grad():
        out = model(fresh_tensor)
        activation_map = cam_extractor(class_idx, out)

    result = overlay_mask(
        to_pil_image(img),
        to_pil_image(activation_map[0].squeeze(0), mode="F"),
        alpha=alpha
    )
    return result

def show_default_layer_comparison(image_paths, titles=None,
                                   target_layer="features.28", save_path=None):
    n = len(image_paths)
    fig, axes = plt.subplots(n, 2, figsize=(12, 4 * n))
    if n == 1:
        axes = [axes]

    for i, image_path in enumerate(image_paths):
        img, input_tensor = load_image(image_path)
        class_idx, confidence, top5 = predict_top5(input_tensor)
        cam_result = make_cam_overlay(img, image_path, class_idx, target_layer)

        label = categories[class_idx]
        custom_title = titles[i] if titles else image_path

        axes[i][0].imshow(to_pil_image(img))
        axes[i][0].set_title(f"{custom_title}\nOriginal")
        axes[i][0].axis("off")

        axes[i][1].imshow(cam_result)
        axes[i][1].set_title(f"{custom_title}\n{label} ({confidence:.1f}%)")
        axes[i][1].axis("off")

        print(f"\n--- {custom_title} ---")
        for rank, (prob, idx) in enumerate(zip(top5.values, top5.indices), start=1):
            print(f"{rank}. {categories[idx]}: {prob.item() * 100:.1f}%")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()

def show_multilayer_analysis(image_path, layers, layer_names, save_path=None):
    img, input_tensor = load_image(image_path)
    class_idx, confidence, top5 = predict_top5(input_tensor)

    fig, axes = plt.subplots(1, len(layers) + 1, figsize=(5 * (len(layers) + 1), 5))

    axes[0].imshow(to_pil_image(img))
    axes[0].set_title(f"Original\n{categories[class_idx]} ({confidence:.1f}%)")
    axes[0].axis("off")

    for i, (layer, layer_name) in enumerate(zip(layers, layer_names), start=1):
        cam_result = make_cam_overlay(img, image_path, class_idx, layer)
        axes[i].imshow(cam_result)
        axes[i].set_title(layer_name)
        axes[i].axis("off")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()

# Introduction

In this notebook I look at interpretability in CNNs, specifically how LayerCAM can show which parts of an image the model actually "looks at" when making a prediction. I'm using VGG16 with pretrained ImageNet weights.
I test the model on a few different images – both cases where the correct class is present and cases where it isn't – and compare how the attribution maps change depending on which convolutional layer you extract them from.

The examples were chosen to illustrate different situations. I use a Tibetan mastiff as a clear positive example and compare it with an image of my own dog Tifa (a Shiba Inu). I also examine a Green mamba compared with a visually similar snake, and finally a CD player compared with a Sega Dreamcast.

## Attribution Maps: Positive and Negative Examples

In [2]:
show_default_layer_comparison(
    image_paths=["data//TM.jpg", "data//tifa.jpg"],
    titles=["Tibetan mastiff (positive)", "Tifa (negative)"],
    target_layer="features.28",
    save_path="figures/tibetan_mastiff_default_comparison.png"
)


--- Tibetan mastiff (positive) ---
1. Tibetan mastiff: 98.3%
2. Leonberg: 0.6%
3. Newfoundland: 0.5%
4. Tibetan terrier: 0.1%
5. collie: 0.1%

--- Tifa (negative) ---
1. Tibetan mastiff: 52.8%
2. malamute: 32.1%
3. Eskimo dog: 5.3%
4. chow: 2.5%
5. Tibetan terrier: 1.6%


<img src="figures/tibetan_mastiff_default_comparison.png" style="width: 50%; height: auto;">

The negative example, my own dog Tifa, a Shiba Inu, was selected partly because she happened to be standing in a pose similar to the mastiff image. This made it interesting to see whether the model would rely on the overall body shape or other distinguishing features.

In practice, the model focuses mostly on the face. The attribution map shows strong activation around the head, while the front paws and body receive only weak attention. This is fairly understandable, since facial features likely contain more distinguishing information for dog breeds.

Interestingly, this differs from the mastiff example, where activation is distributed more broadly across the body, including the shoulders, chest, and legs.

Looking at the logits for the Shiba Inu image reveals that the model is uncertain between several similar dog breeds. While the top prediction is Tibetan mastiff (52.8%), the model also assigns significant probability to breeds such as malamute and Eskimo dog. These breeds share similar visual characteristics such as thick fur and a wolf-like facial structure. This suggests that the model is relying on high-level visual features associated with northern or large spitz-type dogs rather than identifying the exact breed.

In [3]:
show_default_layer_comparison(
    image_paths=["data//GreenMamba.jpg", "data//SpottedBushSnake.jpg"],
    titles=["Green Mamba (positive)", "Spotted Bush Snake(negative)"],
    target_layer="features.28",
    save_path="figures/green_mamba_default_comparison.png"
)


--- Green Mamba (positive) ---
1. green mamba: 98.3%
2. green snake: 0.8%
3. vine snake: 0.4%
4. night snake: 0.1%
5. horned viper: 0.1%

--- Spotted Bush Snake(negative) ---
1. green snake: 47.3%
2. green mamba: 32.5%
3. vine snake: 14.2%
4. night snake: 4.1%
5. thunder snake: 0.5%


<img src="figures/green_mamba_default_comparison.png" style="width: 50%; height: auto;">

In the case of the snake image, the positive example is Green mamba. Attribution maps reveal the strongest activations in the head part, which is quite understandable since the head shape of snakes can serve as one of the distinguishing characteristics. There is some activation in different parts of the body as well, which might be due to the scales' colors and patterns.

For the negative example, I chose another species that looked somewhat alike – Spotted bush snake. The model paid attention to the entire body but mostly focused on the part where spotty scale pattern is prominent. It is intriguing why the model called it "green snake" but not "green mamba"; probably, it confused it due to the spotty scales that made the silhouette less smooth.

However, there is one puzzling activation in the positive example – in the empty space at the bottom-right corner of the picture. It might be caused by the background similarity between some ImageNet training images for the class in question; alternatively, it could be "right answer, wrong reason" – the model did identify the class correctly but based on irrelevant features.

Since the activations on the snake itself are stronger and more consistent overall, they likely dominate the final decision. Still, the corner activation serves as a reminder that high confidence doesn't necessarily mean the model is looking at the right things.

In [4]:
show_default_layer_comparison(
    image_paths=["data//cd.jpg", "data//dreamcast1.jpg"],
    titles=["CD-Player (positive)", "Dreamcast (negative)"],
    target_layer="features.28",
    save_path="figures/cd_player_default_comparison.png"
)

show_default_layer_comparison(
    image_paths=["data//cd.jpg", "data//dreamcast2.jpg"],
    titles=["CD-Player (positive)", "Dreamcast (negative)"],
    target_layer="features.28",
    save_path="figures/cd_player_default_comparison_2.png"
)


--- CD-Player (positive) ---
1. CD player: 94.7%
2. waffle iron: 2.9%
3. modem: 0.7%
4. tape player: 0.4%
5. mouse: 0.3%

--- Dreamcast (negative) ---
1. modem: 59.7%
2. projector: 25.2%
3. CD player: 9.5%
4. printer: 1.9%
5. radio: 1.7%

--- CD-Player (positive) ---
1. CD player: 94.7%
2. waffle iron: 2.9%
3. modem: 0.7%
4. tape player: 0.4%
5. mouse: 0.3%

--- Dreamcast (negative) ---
1. CD player: 93.2%
2. hard disc: 2.5%
3. tape player: 0.9%
4. scale: 0.8%
5. modem: 0.7%


<img src="figures/cd_player_default_comparison.png" style="width: 40%;"> <img src="figures/cd_player_default_comparison_2.png" style="width: 40%;">

For the CD player example I wanted to test something a bit different. The negative example is a Sega Dreamcast, a gaming console I have a soft spot for, which I chose because it shares some obvious visual similarities with a CD player: a round disc tray, similar proportions, and a triangular detail that echoes the eject button. Visually, it is a fairly convincing lookalike.

The model classified the actual CD player correctly at 94.7%, with activation focused around the disc area and control buttons, reasonable features to distinguish a CD player. For the Dreamcast with a dark background it instead predicted CD player at 93.2%, which is a wrong answer but still understandable given the visual similarities.

What got interesting was when I swapped the Dreamcast image for one with a white background. The classification changed completely, dropping to modem at 59.7%, with CD player falling to just 9.5%. Same object, completely different prediction. The attribution map also shifted, now focusing on the ports and connectors along the front rather than the disc area.

This suggests the model is not only responding to the object itself, but is also sensitive to the overall image composition. The darker background may have provided contextual cues that pushed the prediction toward CD player, while the white background appears to shift attention toward other features such as the front connectors. A good reminder that the model sees the whole image, not just the object you put in front of it.

## Layer Analysis

In [5]:
show_multilayer_analysis(
    image_path="data//TM.jpg",
    layers=layers,
    layer_names=layer_names,
    save_path="figures/tibetan_mastiff_multilayer.png"
)

<img src="figures/tibetan_mastiff_multilayer.png" style="width: 75%; height: auto;">

In the early layer the activations are fairly diffuse and mostly follow strong edges and contrasts in the image, especially around the head and outline of the dog. This does not necessarily mean the network already recognizes the face, but those areas contain strong visual gradients that early convolution layers tend to respond to.

In the middle layer the activations start to concentrate more around recognizable parts of the dog. The head is still the strongest region, but the model also begins to respond to areas such as the legs and tail, suggesting that it is starting to assemble larger structural features rather than just edges.

By the late layer the focus becomes more selective. The strongest activations appear around the head and chest, while regions such as the tail receive much less attention. This likely reflects the network narrowing its attention to the parts of the animal that are most useful for distinguishing between dog breeds.

Interestingly, the predicted class itself remains stable across the layers (98.3% Tibetan mastiff). What changes is not the prediction but where the model appears to find the evidence for that decision.

In [6]:
show_multilayer_analysis(
    image_path="data//GreenMamba.jpg",
    layers=layers,
    layer_names=layer_names,
    save_path="figures/green_mamba_multilayer.png"
)

<img src="figures/green_mamba_multilayer.png" style="width: 75%; height: auto;">

I also ran the multilayer analysis on the Green mamba, and the results are noticeably different from the dog example.

In the early layer there is very little activation along the body. Most of the weak responses appear just in front of the head, which likely reflects contrast between the snake and the background rather than clear recognition of the animal itself.

In the middle layer the activation becomes slightly more concentrated, although it remains relatively sparse. The head region starts to stand out more clearly, while the body still shows very limited activity.

By the late layer the network focuses almost entirely on the head. This makes sense, since the head shape is likely one of the most informative features for distinguishing snake species. The rest of the body contributes very little to the final decision.

Compared to the dog example, the activations here are much more localized. In the Mastiff image the network distributed attention across several parts of the body such as the legs, chest, and head. In contrast, the snake example shows a much stronger concentration around a single region. Whether this reflects how snakes are represented in ImageNet or simply properties of this specific image is difficult to say, but the difference is quite noticeable.

In [7]:
show_multilayer_analysis(
    image_path="data//dreamcast2.jpg",
    layers=layers,
    layer_names=layer_names,
    save_path="figures/dreamcast2_multilayer.png"
)

<img src="figures/dreamcast2_multilayer.png" style="width: 75%; height: auto;">

To wrap up the CD player section I also ran the Dreamcast through a multilayer analysis, mostly out of curiosity about what the network actually latches onto.

In the early layer there is not much going on. Weak, scattered activations along some edges and around the disc lid, but nothing that suggests the network has identified any meaningful structure yet. At this stage it mainly responds to simple contrasts and edges.

The middle layer is where things start to become more interesting. Activation begins to concentrate around the Dreamcast logo and the circular disc tray. Looking at the object it is easy to see why: that large circular lid visually resembles the top of a portable CD player, which is likely contributing to the model predicting that class.

By the late layer the activations spread across larger parts of the disc area and the upper edges of the console. This suggests the model is responding to the overall circular geometry and layout of the device rather than a single distinctive feature.

Since the Dreamcast itself is not part of the ImageNet classes, the model effectively maps it to the closest visual category it knows. In this case, a device with a large circular lid ends up resembling a CD player closely enough to produce a high-confidence misclassification (93.2%).

## Conclusion

Overall, the attribution maps give a useful glimpse into how the model arrives at its predictions. Early convolution layers mostly react to simple visual patterns such as edges and contrasts, while deeper layers focus more on the parts of the object that are most informative for classification.

The different examples also highlight how the model handles both clear cases and more ambiguous ones. When the correct class is present, the network tends to focus on distinctive features such as the face of the dog or the head of the snake. When the object is unfamiliar, as with the Dreamcast, the model instead maps it to the closest known class based on visual similarity.

While attribution methods like LayerCAM do not fully explain how the network works internally, they provide a useful way to visualize what parts of the image influence the prediction, making the model’s behavior easier to interpret.

## References
Simonyan, K., & Zisserman, A. (2015).
Very Deep Convolutional Networks for Large-Scale Image Recognition.
ICLR.